# BERT Notebook 2/3: Partial Fine-tuning (frozen base, unfrozen pooler)

Second part of the split project. Here we only train BERT's pooler layer (the base is
frozen).

## Google Drive: mounting and paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Must match the path used in Notebook 1
PROJECT_ROOT = '/content/drive/MyDrive/quora_dup_detection'
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
ARTIFACTS_DIR = os.path.join(PROJECT_ROOT, 'artifacts')
CHECKPOINTS_DIR = os.path.join(PROJECT_ROOT, 'checkpoints', 'partial_finetune')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

print('DATA_DIR:       ', DATA_DIR)
print('ARTIFACTS_DIR:  ', ARTIFACTS_DIR)
print('CHECKPOINTS_DIR:', CHECKPOINTS_DIR)

Mounted at /content/drive
DATA_DIR:        /content/drive/MyDrive/quora_dup_detection/data
ARTIFACTS_DIR:   /content/drive/MyDrive/quora_dup_detection/artifacts
CHECKPOINTS_DIR: /content/drive/MyDrive/quora_dup_detection/checkpoints/partial_finetune


## Imports and GPU check

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import torch
from sklearn.metrics import log_loss, accuracy_score, f1_score, roc_auc_score

from datasets import Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cpu':
    print('WARNING: no GPU detected! Runtime -> Change runtime type -> GPU.')
else:
    print(torch.cuda.get_device_name(0))

Device: cuda
Tesla T4


## Loading data from Google Drive

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_features_full.csv'))
val_df = pd.read_csv(os.path.join(DATA_DIR, 'val_features_full.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_features_full.csv'))

print('Train shape:', train_df.shape)
print('Val shape:  ', val_df.shape)
print('Test shape: ', test_df.shape)

y_train = train_df['is_duplicate'].values
y_val = val_df['is_duplicate'].values
y_test = test_df['is_duplicate'].values

# results_train.csv / results_val.csv here are already the updated version from Notebook 1
# (contains TF-IDF/Word2Vec + LogReg/XGBoost on BERT embeddings)
train_results_prev = pd.read_csv(os.path.join(DATA_DIR, 'results_train_p2.csv'))
val_results_prev = pd.read_csv(os.path.join(DATA_DIR, 'results_val_p2.csv'))
val_results_prev

Train shape: (266488, 22)
Val shape:   (56941, 22)
Test shape:  (80858, 22)


,model,log_loss,f1,accuracy,roc_auc
0,Naive constant baseline,0.672988,0.000000,0.605065,0.500000
1,Threshold on word_share_jaccard,0.650177,0.655511,0.636659,0.722330
2,Logistic Regression (TF-IDF + features),0.525917,0.581317,0.711052,0.804393
3,XGBoost (TF-IDF),0.521324,0.523074,0.699988,0.797677
4,Logistic Regression (Word2Vec),0.531765,0.581234,0.700198,0.790394
5,XGBoost (Word2Vec),0.489078,0.630928,0.737992,0.829651
6,"Logistic Regression (BERT, frozen)",0.502090,0.636392,0.738027,0.826067
7,"XGBoost (BERT, frozen)",0.464834,0.669767,0.761806,0.852425


## Loading the model and tokenizer

In [ ]:
model_path = 'google-bert/bert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(model_path)

id2label = {0: 'Not Duplicate', 1: 'Duplicate'}
label2id = {'Not Duplicate': 0, 'Duplicate': 1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_path, num_labels=2, id2label=id2label, label2id=label2id
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
model.eval()
inputs = tokenizer(
    "how do i learn python",
    "what is the best way to learn python",
    return_tensors="pt"
)

with torch.no_grad():
    logits = model(**inputs).logits

print("Logits of the untrained model (should be close to random):", logits)

Logits of the untrained model (should be close to random): tensor([[-0.4411,  0.6221]])


## Preparing the model: freeze the base, unfreeze the pooler

In [ ]:
# Freeze the parameters of the base model
for name, param in model.base_model.named_parameters():
    param.requires_grad = False

# Unfreeze the pooling layer, which is used for classification
for name, param in model.base_model.named_parameters():
    if "pooler" in name:
        param.requires_grad = True

trainable_params = [
    name for name, param in model.named_parameters() if param.requires_grad
]
total_params = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Trainable parameters:", trainable_params)
print(
    f"Training {trainable_count:,} out of {total_params:,} parameters "
    f"({trainable_count / total_params * 100:.2f}%)"
)

Trainable parameters: ['bert.pooler.dense.weight', 'bert.pooler.dense.bias', 'classifier.weight', 'classifier.bias']
Training 592,130 out of 109,483,778 parameters (0.54%)


## `datasets.Dataset` and tokenizing question pairs

We encode question pairs. `tokenizer(examples["question1"], examples["question2"], truncation=True)`
treats each pair as a single input sequence.

In [ ]:
dataset_dict = DatasetDict({
    'train': Dataset.from_pandas(train_df[['question1', 'question2', 'is_duplicate']].reset_index(drop=True)),
    'val': Dataset.from_pandas(val_df[['question1', 'question2', 'is_duplicate']].reset_index(drop=True)),
    'test': Dataset.from_pandas(test_df[['question1', 'question2', 'is_duplicate']].reset_index(drop=True)),
})
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['question1', 'question2', 'is_duplicate'],
        num_rows: 266488
    })
    val: Dataset({
        features: ['question1', 'question2', 'is_duplicate'],
        num_rows: 56941
    })
    test: Dataset({
        features: ['question1', 'question2', 'is_duplicate'],
        num_rows: 80858
    })
})

In [ ]:
def preprocess_function(examples):
    return tokenizer(
        examples["question1"],
        examples["question2"],
        truncation=True,
        max_length=64,
    )

tokenized_data = dataset_dict.map(preprocess_function, batched=True)
tokenized_data = tokenized_data.rename_column("is_duplicate", "labels")

print(tokenized_data)
print()
print("Example of a tokenized sample:", tokenized_data["train"][0])

Map:   0%|          | 0/266488 [00:00<?, ? examples/s]

Map:   0%|          | 0/56941 [00:00<?, ? examples/s]

Map:   0%|          | 0/80858 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question1', 'question2', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 266488
    })
    val: Dataset({
        features: ['question1', 'question2', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 56941
    })
    test: Dataset({
        features: ['question1', 'question2', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 80858
    })
})

Example of a tokenized sample: {'question1': 'The Iliad and the Odyssey in the Greek culture?', 'question2': 'How do I prove that the pairs of three independent variables is also independent?', 'labels': 0, 'input_ids': [101, 1996, 6335, 28665, 1998, 1996, 18735, 1999, 1996, 3306, 3226, 1029, 102, 2129, 2079, 1045, 6011, 2008, 1996, 7689, 1997, 2093, 2981, 10857, 2003, 2036, 2981, 1029, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Training the model

In [ ]:
lr = 2e-4
batch_size = 32
num_epochs = 3

training_args = TrainingArguments(
    output_dir=CHECKPOINTS_DIR,
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    logging_strategy='epoch',
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model='log_loss',
    greater_is_better=False,
    report_to=[],
)

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    probabilities = np.exp(predictions) / np.exp(predictions).sum(-1, keepdims=True)
    positive_class_probs = probabilities[:, 1]
    predicted_classes = np.argmax(predictions, axis=1)

    return {
        'accuracy': accuracy_score(labels, predicted_classes),
        'auc': roc_auc_score(labels, positive_class_probs),
        'log_loss': log_loss(labels, positive_class_probs, labels=[0, 1]),
        'f1': f1_score(labels, predicted_classes),
    }

In [ ]:
%%time
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data['train'],
    eval_dataset=tokenized_data['val'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Auc,Log Loss,F1,Runtime,Samples Per Second,Steps Per Second
1,0.488202,0.492427,0.735867,0.824209,0.492428,0.639916,56.627800,1005.531000,31.433000
2,0.466633,0.484882,0.746949,0.828456,0.484882,0.691463,55.631000,1023.548000,31.997000
3,0.457604,0.484528,0.746755,0.829451,0.484529,0.699558,55.349800,1028.749000,32.159000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

CPU times: user 17min 47s, sys: 8.34 s, total: 17min 55s
Wall time: 18min 52s


TrainOutput(global_step=24984, training_loss=0.4708126286761027, metrics={'train_runtime': 1130.9144, 'train_samples_per_second': 706.918, 'train_steps_per_second': 22.092, 'total_flos': 2.457679668126384e+16, 'train_loss': 0.4708126286761027, 'epoch': 3.0})

## Evaluation on train and validation

In [ ]:
%%time
# Train
train_preds_bert_ft = trainer.predict(tokenized_data['train'])

train_proba_bert_ft = np.exp(train_preds_bert_ft.predictions)
train_proba_bert_ft /= train_proba_bert_ft.sum(axis=1, keepdims=True)
train_proba_bert_ft = train_proba_bert_ft[:, 1]

# Validation
val_preds_bert_ft = trainer.predict(tokenized_data['val'])

val_proba_bert_ft = np.exp(val_preds_bert_ft.predictions)
val_proba_bert_ft /= val_proba_bert_ft.sum(axis=1, keepdims=True)
val_proba_bert_ft = val_proba_bert_ft[:, 1]

CPU times: user 5min 10s, sys: 1.16 s, total: 5min 11s
Wall time: 5min 25s


In [ ]:
def add_result(results, model_name, y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)

    row = {
        'model': model_name,
        'log_loss': log_loss(y_true, y_proba),
        'f1': f1_score(y_true, y_pred),
        'accuracy': accuracy_score(y_true, y_pred),
        'roc_auc': roc_auc_score(y_true, y_proba)
    }

    results.loc[len(results)] = row
    display(results)

    return results

In [ ]:
print('='*30+'Train'+'='*30)
train_results = add_result(
    train_results_prev,
    'BERT (fine-tuned, frozen base)',
    y_train,
    train_proba_bert_ft
)

print('='*30+'Validation'+'='*30)
val_results = add_result(
    val_results_prev,
    'BERT (fine-tuned, frozen base)',
    y_val,
    val_proba_bert_ft
)

==============================Train==============================


,model,log_loss,f1,accuracy,roc_auc
0,Naive constant baseline,0.655518,0.000000,0.636299,0.500000
1,Threshold on word_share_jaccard,0.648484,0.632415,0.625364,0.732250
2,Logistic Regression (TF-IDF + features),0.426158,0.697843,0.790347,0.873204
3,XGBoost (TF-IDF),0.448985,0.653174,0.774545,0.858775
4,Logistic Regression (Word2Vec),0.492585,0.623092,0.735035,0.815719
5,XGBoost (Word2Vec),0.360521,0.780661,0.842781,0.922449
6,"Logistic Regression (BERT, frozen)",0.425666,0.705789,0.789011,0.870666
7,"XGBoost (BERT, frozen)",0.325103,0.808335,0.860969,0.938558
8,"BERT (fine-tuned, frozen base)",0.433724,0.739224,0.783773,0.875156


==============================Validation==============================


,model,log_loss,f1,accuracy,roc_auc
0,Naive constant baseline,0.672988,0.000000,0.605065,0.500000
1,Threshold on word_share_jaccard,0.650177,0.655511,0.636659,0.722330
2,Logistic Regression (TF-IDF + features),0.525917,0.581317,0.711052,0.804393
3,XGBoost (TF-IDF),0.521324,0.523074,0.699988,0.797677
4,Logistic Regression (Word2Vec),0.531765,0.581234,0.700198,0.790394
5,XGBoost (Word2Vec),0.489078,0.630928,0.737992,0.829651
6,"Logistic Regression (BERT, frozen)",0.502090,0.636392,0.738027,0.826067
7,"XGBoost (BERT, frozen)",0.464834,0.669767,0.761806,0.852425
8,"BERT (fine-tuned, frozen base)",0.484529,0.699558,0.746755,0.829451


**Conclusion:** Fine-tuning just one small layer of BERT (partial fine-tuning) performed worse than simply freezing BERT entirely and training a separate classifier on top. That's because the rest of BERT — the part that actually understands the text — was never adjusted for this specific task. One small layer can't fix that on its own. This is why the next step is full fine-tuning: unfreezing the entire model so it can truly learn what makes two questions duplicates.

## Saving updated metric tables to Drive

In [ ]:
train_results.to_csv(os.path.join(DATA_DIR, 'results_train_p3.csv'), index=False)
val_results.to_csv(os.path.join(DATA_DIR, 'results_val.csv_p3'), index=False)
print('Updated results_train_p3.csv / results_val_p3.csv saved to Drive.')

Updated results_train_p3.csv / results_val_p3.csv saved to Drive.
